## Outlier Analysis & Handling

In [2]:
import pandas as pd

listings = pd.read_pickle("..\\processed\\CRMLSListings_Engineered.pkl")
sold = pd.read_pickle("..\\processed\\CRMLSSold_Engineered.pkl")

Form outlier ranges and flag outliers

In [ ]:
metrics = {"ClosePrice": "ClosePriceOutlier", "LivingArea": "LivingAreaOutlier",
           "Price Per Sq Ft": "PricePerSqFtOutlier", "Price Ratio": "PriceRatioOutlier",
           "Days on Market": "DaysOnMarketOutlier", "Close to Original List Ratio": "CloseToOriginalListPriceOutlier"}

for metric, outlier_metric in metrics.items():
    Q1 = listings[metric].quantile(0.25)
    Q3 = listings[metric].quantile(0.75)
    IQR = Q3 - Q1
    listings[outlier_metric] = (listings[metric] < Q1 - 1.5 * IQR) | (listings[metric] > Q3 + 1.5 * IQR)
    sold[outlier_metric] = (sold[metric] < Q1 - 1.5 * IQR) | (sold[metric] > Q3 + 1.5 * IQR)

for name, df in [("Sold", sold), ("Listings", listings)]:
    df["AnyOutlier"] = df[list(metrics.values())].any(axis=1)
    print(f"\n{name} outliers:")
    any_outliers = df["AnyOutlier"].sum()
    print(
        f"AnyOutlier: {any_outliers:,} / {len(df):,} "
        f"({any_outliers / len(df) * 100:.2f}%)"
    )

    for outlier_metric in metrics.values():
        matched = df[outlier_metric].sum()
        print(
            f"{outlier_metric}: {matched:,} / {len(df):,} "
            f"({matched / len(df) * 100:.2f}%)"
        )


Sold outliers:
AnyOutlier: 184,813 / 480,314 (38.48%)
ClosePriceOutlier: 33,042 / 480,314 (6.88%)
LivingAreaOutlier: 17,137 / 480,314 (3.57%)
PricePerSqFtOutlier: 18,841 / 480,314 (3.92%)
PriceRatioOutlier: 52,520 / 480,314 (10.93%)
DaysOnMarketOutlier: 112,478 / 480,314 (23.42%)
CloseToOriginalListPriceOutlier: 56,496 / 480,314 (11.76%)

Listings outliers:
AnyOutlier: 123,206 / 644,002 (19.13%)
ClosePriceOutlier: 13,886 / 644,002 (2.16%)
LivingAreaOutlier: 31,612 / 644,002 (4.91%)
PricePerSqFtOutlier: 8,741 / 644,002 (1.36%)
PriceRatioOutlier: 22,367 / 644,002 (3.47%)
DaysOnMarketOutlier: 66,357 / 644,002 (10.30%)
CloseToOriginalListPriceOutlier: 19,849 / 644,002 (3.08%)


### Save flagged dataset

In [9]:
sold.to_pickle("..\\processed\\CRMLSSold_OutlierFlagged.pkl")
listings.to_pickle("..\\processed\\CRMLSListings_OutlierFlagged.pkl")

sold.to_csv("..\\processed\\CRMLSSold_OutlierFlagged.csv")
listings.to_csv("..\\processed\\CRMLSListings_OutlierFlagged.csv")